In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.signal import butter, filtfilt, find_peaks, hilbert
import warnings
import re

In [5]:
# ====== PARAMETRES ======
dossier = "Coda_data"

fichiers_S01 = [
    "Gr4_S04_S1_20250011_013.txt",
    "Gr4_S04_S1_20250011_014.txt",
    "Gr4_S04_S1_20250011_015.txt",
    "Gr4_S04_S2_20250011_016.txt",
    "Gr4_S04_S2_20250011_017.txt",
    "Gr4_S04_S2_20250011_018.txt",
    "Gr4_S04_S3_20250011_019.txt",
    "Gr4_S04_S3_20250011_020.txt",
    "Gr4_S04_S3_20250011_021.txt",
    "Gr4_S04_S4_20250011_022.txt",
    "Gr4_S04_S4_20250011_023.txt",
    "Gr4_S04_S4_20250011_024.txt",
]

# ====== FILTRE PASSE-BAS ======
fs = 200
fc = 5
b, a = butter(4, fc/(fs/2), btype='low')
h = 1/fs  # pas de temps = 0.005 s

# ====== DATAFRAME FINAL ======
resultats = []

# ====== DATAFRAME POUR ANGLE ======
resultats_angle = []

for f in fichiers_S01:
    chemin = os.path.join(dossier, f)

    df = pd.read_csv(chemin, sep="\t", header=2)
    df.columns = df.columns.str.strip()
    df = df.replace("NaN", np.nan)

    colonnes_a_supprimer = [c for c in df.columns if ".V" in c or c.strip() == ""]
    df = df.drop(columns=colonnes_a_supprimer, errors="ignore")

    time_col = "Time" if "Time" in df.columns else "Time (s)"

    pattern = r"Marker (1[7-9]|20)\.[XYZ]"
    
    colonnes_markers = [c for c in df.columns if re.search(pattern, c)]
    df = df[[time_col] + colonnes_markers]
    df = df.apply(pd.to_numeric, errors="coerce")

    # ====== CENTRE RECTANGLE ======
    cx = ((df["Marker 17.X"] + df["Marker 18.X"])/2 +
          (df["Marker 19.X"] + df["Marker 20.X"])/2)/2

    cy = ((df["Marker 17.Y"] + df["Marker 18.Y"])/2 +
          (df["Marker 19.Y"] + df["Marker 20.Y"])/2)/2

    cz = ((df["Marker 17.Z"] + df["Marker 18.Z"])/2 +
          (df["Marker 19.Z"] + df["Marker 20.Z"])/2)/2

    # ====== MILIEU MARKERS 5-6 ======
    top_x = (df["Marker 17.X"] + df["Marker 18.X"]) / 2
    top_z = (df["Marker 17.Y"] + df["Marker 18.Y"]) / 2

    # ====== INTERPOLATION NaN ======
    for signal in [cx, cy, cz, top_x, top_z]:
        nan_idx = np.isnan(signal)
        if np.any(nan_idx):
            signal[nan_idx] = np.interp(
                np.flatnonzero(nan_idx),
                np.flatnonzero(~nan_idx),
                signal[~nan_idx]
            )

    # ====== FILTRAGE ======
    cx = filtfilt(b, a, cx)
    cy = filtfilt(b, a, cy)
    cz = filtfilt(b, a, cz)

    top_x = filtfilt(b, a, top_x)
    top_z = filtfilt(b, a, top_z)

    # ====== NOM PROPRE ======
    m = re.search(r"Gr4_(S\d+)_(S\d+)_\d+_(\d+)\.txt$", f)
    if m:
        s_part = m.group(1)
        l_part = m.group(2)
        num = str(int(m.group(3)))
        col_base = f"{s_part}_{l_part}_{num}"
    else:
        col_base = f.replace(".txt","")

    # ====== DATAFRAME POSITION (comme avant) ======
    df_res = pd.DataFrame({
        "Time": df[time_col],
        f"{col_base}_X": cx,
        f"{col_base}_Y": cy,
        f"{col_base}_Z": cz
    })

    resultats.append(df_res)

    # ====== DATAFRAME POUR ANGLE ======
    df_angle = pd.DataFrame({
        "Time": df[time_col],
        f"{col_base}_center_X": cx,
        f"{col_base}_center_Y": cy,
        f"{col_base}_top_X": top_x,
        f"{col_base}_top_Z": top_z
    })

    resultats_angle.append(df_angle)


# ====== FUSION DATAFRAME POSITION ======
df_final = resultats[0]
for df in resultats[1:]:
    df_final = df_final.merge(df, on="Time")

# ====== FUSION DATAFRAME ANGLE ======
df_angle_base = resultats_angle[0]
for df in resultats_angle[1:]:
    df_angle_base = df_angle_base.merge(df, on="Time")

# ====== AFFICHAGE ======
print("✅ Fusion terminée")

print("\nColonnes df_final :")
print(df_final.columns.tolist())

print("\nColonnes df_angle_base :")
print(df_angle_base.columns.tolist())

✅ Fusion terminée

Colonnes df_final :
['Time', 'S04_S1_13_X', 'S04_S1_13_Y', 'S04_S1_13_Z', 'S04_S1_14_X', 'S04_S1_14_Y', 'S04_S1_14_Z', 'S04_S1_15_X', 'S04_S1_15_Y', 'S04_S1_15_Z', 'S04_S2_16_X', 'S04_S2_16_Y', 'S04_S2_16_Z', 'S04_S2_17_X', 'S04_S2_17_Y', 'S04_S2_17_Z', 'S04_S2_18_X', 'S04_S2_18_Y', 'S04_S2_18_Z', 'S04_S3_19_X', 'S04_S3_19_Y', 'S04_S3_19_Z', 'S04_S3_20_X', 'S04_S3_20_Y', 'S04_S3_20_Z', 'S04_S3_21_X', 'S04_S3_21_Y', 'S04_S3_21_Z', 'S04_S4_22_X', 'S04_S4_22_Y', 'S04_S4_22_Z', 'S04_S4_23_X', 'S04_S4_23_Y', 'S04_S4_23_Z', 'S04_S4_24_X', 'S04_S4_24_Y', 'S04_S4_24_Z']

Colonnes df_angle_base :
['Time', 'S04_S1_13_center_X', 'S04_S1_13_center_Y', 'S04_S1_13_top_X', 'S04_S1_13_top_Z', 'S04_S1_14_center_X', 'S04_S1_14_center_Y', 'S04_S1_14_top_X', 'S04_S1_14_top_Z', 'S04_S1_15_center_X', 'S04_S1_15_center_Y', 'S04_S1_15_top_X', 'S04_S1_15_top_Z', 'S04_S2_16_center_X', 'S04_S2_16_center_Y', 'S04_S2_16_top_X', 'S04_S2_16_top_Z', 'S04_S2_17_center_X', 'S04_S2_17_center_Y', 'S04_

In [10]:
# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_plots = os.path.join(dossier_root, "position_S04_filtre")
os.makedirs(dossier_plots, exist_ok=True)

time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

# ====== PLOTS ======
for col in cols:

    plt.figure(figsize=(8,3))

    plt.plot(df_final[time_col], df_final[col])

    plt.xlabel("Temps (s)")
    plt.ylabel(col)
    plt.title(f"{col} filtré (<{fc} Hz)")

    ticks = np.arange(
        0,
        float(df_final[time_col].iloc[-1]) + 1,
        5
    )
    plt.xticks(ticks)

    plt.tight_layout()

    # sauvegarde
    nom_fichier = f"{col}.png"
    chemin_plot = os.path.join(dossier_plots, nom_fichier)

    plt.savefig(chemin_plot, dpi=150)
    plt.close()

print(f"✅ Plots sauvegardés dans : {dossier_plots}")

✅ Plots sauvegardés dans : Cinematique_S04_S\position_S04_filtre


In [11]:
# ====== FONCTION VITESSE CENTREE ======
def vitesse_centre(U, h):
    """
    Dérivée centrée d'ordre 4 :
    u'[i] = (-U[i+2] + 8*U[i+1] - 8*U[i-1] + U[i-2]) / (12*h)
    """
    U = np.array(U, dtype=float)
    N = len(U)
    V = np.zeros_like(U)

    # début
    V[0] = (U[1] - U[0]) / h
    V[1] = (U[2] - U[1]) / h

    # points centraux
    for i in range(2, N-2):
        V[i] = (-U[i+2] + 8*U[i+1] - 8*U[i-1] + U[i-2]) / (12*h)

    # fin
    V[-2] = (U[-2] - U[-3]) / h
    V[-1] = (U[-1] - U[-2]) / h

    return V


# ====== CALCUL DES VITESSES ======

velocities = pd.DataFrame()

time_col = "Time"
velocities[time_col] = df_final[time_col]

cols = [c for c in df_final.columns if c != time_col]

for col in cols:

    velocities[col] = vitesse_centre(
        df_final[col].values,
        h
    )

print("✅ Calcul des vitesses terminé")

print("\nColonnes vitesses :")
print(velocities.columns.tolist())

✅ Calcul des vitesses terminé

Colonnes vitesses :
['Time', 'S04_S1_13_X', 'S04_S1_13_Y', 'S04_S1_13_Z', 'S04_S1_14_X', 'S04_S1_14_Y', 'S04_S1_14_Z', 'S04_S1_15_X', 'S04_S1_15_Y', 'S04_S1_15_Z', 'S04_S2_16_X', 'S04_S2_16_Y', 'S04_S2_16_Z', 'S04_S2_17_X', 'S04_S2_17_Y', 'S04_S2_17_Z', 'S04_S2_18_X', 'S04_S2_18_Y', 'S04_S2_18_Z', 'S04_S3_19_X', 'S04_S3_19_Y', 'S04_S3_19_Z', 'S04_S3_20_X', 'S04_S3_20_Y', 'S04_S3_20_Z', 'S04_S3_21_X', 'S04_S3_21_Y', 'S04_S3_21_Z', 'S04_S4_22_X', 'S04_S4_22_Y', 'S04_S4_22_Z', 'S04_S4_23_X', 'S04_S4_23_Y', 'S04_S4_23_Z', 'S04_S4_24_X', 'S04_S4_24_Y', 'S04_S4_24_Z']


In [12]:
# ===== TEMPS A OBSERVER ========
t_min = 7
t_max = 39

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_plots_vitesse = os.path.join(dossier_root, "vitesse_S04")
os.makedirs(dossier_plots_vitesse, exist_ok=True)

# ====== PLOT DES VITESSES (+timing) ======
mask = (velocities[time_col] >= t_min) & (velocities[time_col] <= t_max)

for col in cols:
    plt.figure(figsize=(8,3))
    plt.plot(velocities[time_col][mask], velocities[col][mask], color='blue')
    plt.xlabel("Temps (s)")
    plt.ylabel(f"Vitesse de {col}")
    plt.title(f"Vitesse centrée filtrée (<{fc} Hz) de {col} ({t_min}–{t_max} s)")
    plt.grid(True)
    plt.tight_layout()

    # sauvegarde
    nom_fichier = f"vitesse_{col.replace(' ', '_')}.png"
    chemin_plot = os.path.join(dossier_plots_vitesse, nom_fichier)
    plt.savefig(chemin_plot, dpi=150)
    plt.close()

print(f"✅ Plot terminé et sauvegardé dans \"{dossier_plots_vitesse}\"")

✅ Plot terminé et sauvegardé dans "Cinematique_S04_S\vitesse_S04"


In [13]:
# ===== TEMPS A OBSERVER ========
t_min = 7
t_max = 39


# ===== PARAMETRES ==========
distance_pts = 200
prominence_pts = 5

time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

# ====== SELECTION INTERVALLE TEMPS ======
mask = (df_final[time_col] >= t_min) & (df_final[time_col] <= t_max)
time_segment = df_final[time_col][mask]

# ====== STOCKAGE ======
all_min_times = {}
all_max_times = {}

for col in cols:

    signal = df_final[col][mask].values
    time_vals = time_segment.values

    # ====== DETECTION MINIMA ======
    peaks_min, _ = find_peaks(
        -signal,
        distance=distance_pts,
        prominence=prominence_pts
    )

    minima_times = list(time_vals[peaks_min])

    # enlever premier et dernier
    if len(minima_times) > 2:
        minima_times = minima_times[1:-1]

    all_min_times[col] = minima_times


    # ====== DETECTION MAXIMA ======
    peaks_max, _ = find_peaks(
        signal,
        distance=distance_pts,
        prominence=prominence_pts
    )

    maxima_times = list(time_vals[peaks_max])

    # enlever premier et dernier
    if len(maxima_times) > 2:
        maxima_times = maxima_times[1:-1]

    all_max_times[col] = maxima_times


    # ====== PRINT INFO ======
    print(f"\n📊 {col}")
    print(f"Minima détectés (après suppression bords) : {len(minima_times)}")
    print(f"Maxima détectés (après suppression bords) : {len(maxima_times)}")


# ====== DATAFRAME MINIMA ======
max_len_min = max(len(v) for v in all_min_times.values())

minima_times_df = pd.DataFrame({
    col: pd.Series(v + [np.nan]*(max_len_min - len(v)))
    for col, v in all_min_times.items()
})


# ====== DATAFRAME MAXIMA ======
max_len_max = max(len(v) for v in all_max_times.values())

maxima_times_df = pd.DataFrame({
    col: pd.Series(v + [np.nan]*(max_len_max - len(v)))
    for col, v in all_max_times.items()
})


print(f"\n✅ DataFrame des minima : {minima_times_df.shape[0]} x {minima_times_df.shape[1]}")
print(f"✅ DataFrame des maxima : {maxima_times_df.shape[0]} x {maxima_times_df.shape[1]}")


📊 S04_S1_13_X
Minima détectés (après suppression bords) : 13
Maxima détectés (après suppression bords) : 14

📊 S04_S1_13_Y
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 15

📊 S04_S1_13_Z
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 13

📊 S04_S1_14_X
Minima détectés (après suppression bords) : 13
Maxima détectés (après suppression bords) : 14

📊 S04_S1_14_Y
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 14

📊 S04_S1_14_Z
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 14

📊 S04_S1_15_X
Minima détectés (après suppression bords) : 17
Maxima détectés (après suppression bords) : 14

📊 S04_S1_15_Y
Minima détectés (après suppression bords) : 16
Maxima détectés (après suppression bords) : 14

📊 S04_S1_15_Z
Minima détectés (après suppression bords) : 14
Maxima détectés (après suppression bords) : 13

📊 S04_S2_16_X
Mini

In [14]:
# ===== TEMPS A OBSERVER ========
t_min = 7
t_max = 39

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_plots_vitesse = os.path.join(dossier_root, "vitesse_S04_extrema")
os.makedirs(dossier_plots_vitesse, exist_ok=True)

# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in velocities.columns if c != time_col]

print(f"Velocities DataFrame : {velocities.shape[0]} lignes x {velocities.shape[1]} colonnes")
print(f"Minima Times DataFrame : {minima_times_df.shape[0]} lignes x {minima_times_df.shape[1]} colonnes")
print(f"Maxima Times DataFrame : {maxima_times_df.shape[0]} lignes x {maxima_times_df.shape[1]} colonnes\n")


for col in cols:

    # ====== SELECTION DE L'INTERVALLE ======
    mask = (velocities[time_col] >= t_min) & (velocities[time_col] <= t_max)

    time_vals = velocities[time_col][mask].values
    signal = velocities[col][mask].values

    plt.figure(figsize=(8,3))
    plt.plot(time_vals, signal, color='blue', label="Vitesse")


    # ====== MINIMA ======
    if col in minima_times_df.columns:

        peak_times = minima_times_df[col].dropna().values
        peak_times = peak_times[(peak_times >= t_min) & (peak_times <= t_max)]

        if len(peak_times) > 0:

            peak_indices = []
            for t in peak_times:
                idx = np.argmin(np.abs(time_vals - t))
                peak_indices.append(idx)

            peak_values = signal[peak_indices]

            plt.scatter(time_vals[peak_indices], peak_values,
                        color='red', s=30, label="Minima")


    # ====== MAXIMA ======
    if col in maxima_times_df.columns:

        peak_times = maxima_times_df[col].dropna().values
        peak_times = peak_times[(peak_times >= t_min) & (peak_times <= t_max)]

        if len(peak_times) > 0:

            peak_indices = []
            for t in peak_times:
                idx = np.argmin(np.abs(time_vals - t))
                peak_indices.append(idx)

            peak_values = signal[peak_indices]

            plt.scatter(time_vals[peak_indices], peak_values,
                        color='green', s=30, label="Maxima")


    # ====== FIGURE ======
    plt.xlabel("Temps (s)")
    plt.ylabel(f"Vitesse {col}")
    plt.title(f"Vitesse filtrée (<{fc} Hz) de {col} ({t_min}-{t_max}s)")
    plt.grid(True)
    plt.legend()
    plt.xlim(t_min, t_max)
    plt.tight_layout()


    # ====== SAUVEGARDE ======
    nom_fichier = f"vitesse_{col}.png"
    chemin_plot = os.path.join(dossier_plots_vitesse, nom_fichier)

    plt.savefig(chemin_plot, dpi=150)
    plt.close()


print(f"\n✅ Plots vitesse sauvegardés dans : {dossier_plots_vitesse}")

Velocities DataFrame : 8201 lignes x 37 colonnes
Minima Times DataFrame : 17 lignes x 36 colonnes
Maxima Times DataFrame : 15 lignes x 36 colonnes


✅ Plots vitesse sauvegardés dans : Cinematique_S04_S\vitesse_S04_extrema


In [15]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_plots_superp = os.path.join(dossier_root, "superp_S04")
os.makedirs(dossier_plots_superp, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION DES CYCLES ======
for col in cols:

    if col not in minima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values

    if len(minima_times) < 2:
        continue

    fig, ax = plt.subplots(figsize=(8,4))

    n_cycles = len(minima_times) - 1

    for i in range(n_cycles):

        t_start = minima_times[i]
        t_end = minima_times[i+1]

        mask = (df_final[time_col] >= t_start) & (df_final[time_col] <= t_end)

        segment_time = df_final[time_col][mask].values - t_start
        segment_signal = df_final[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_cycles)

        ax.plot(segment_time, segment_signal, color=color)

    ax.set_xlabel("Temps depuis minima (s)")
    ax.set_ylabel(col)
    ax.set_title(f"Superposition des cycles : {col}")
    ax.grid(True)

    # ====== COLORBAR ======
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=n_cycles))
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Numéro du cycle\n(premiers → derniers)")

    fig.tight_layout()

    # ====== SAUVEGARDE ======
    nom_fichier = f"superp_{col}.png"
    chemin_plot = os.path.join(dossier_plots_superp, nom_fichier)

    fig.savefig(chemin_plot, dpi=150)
    plt.close(fig)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_plots_superp}")


✅ Graphiques sauvegardés dans : Cinematique_S04_S\superp_S04


In [16]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in velocities.columns if c != time_col]

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_plots_superv = os.path.join(dossier_root, "superv_S04")
os.makedirs(dossier_plots_superv, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION DES VITESSES PAR CYCLE ======
for col in cols:

    if col not in minima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values

    if len(minima_times) < 2:
        continue

    fig, ax = plt.subplots(figsize=(8,4))

    n_cycles = len(minima_times) - 1

    for i in range(n_cycles):

        t_start = minima_times[i]
        t_end = minima_times[i+1]

        # intervalle du cycle
        mask = (velocities[time_col] >= t_start) & (velocities[time_col] <= t_end)

        segment_time = velocities[time_col][mask].values - t_start
        segment_signal = velocities[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_cycles)

        ax.plot(segment_time, segment_signal, color=color)

    ax.set_xlabel("Temps depuis minima (s)")
    ax.set_ylabel(f"Vitesse {col}")
    ax.set_title(f"Superposition des vitesses : {col}")

    ax.grid(True)

    # ====== COLORBAR ======
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=n_cycles))
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Numéro du cycle\n(premiers → derniers)")

    fig.tight_layout()

    # ====== SAUVEGARDE ======
    nom_fichier = f"superv_{col}.png"
    chemin_plot = os.path.join(dossier_plots_superv, nom_fichier)

    fig.savefig(chemin_plot, dpi=150)
    plt.close(fig)

print(f"\n✅ Graphiques de vitesses sauvegardés dans : {dossier_plots_superv}")


✅ Graphiques de vitesses sauvegardés dans : Cinematique_S04_S\superv_S04


In [17]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_plots_sep = os.path.join(dossier_root, "superp_sep_S04")
os.makedirs(dossier_plots_sep, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION ASCENDANT / DESCENDANT ======
for col in cols:

    if col not in minima_times_df.columns or col not in maxima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values
    maxima_times = maxima_times_df[col].dropna().values

    if len(minima_times) < 1 or len(maxima_times) < 1:
        continue

    # ====== FIGURES ======
    fig_up, ax_up = plt.subplots(figsize=(8,4))
    fig_down, ax_down = plt.subplots(figsize=(8,4))

    cycles_up = []
    cycles_down = []

    # ====== ASSOCIATION MIN/MAX ======
    for t_min in minima_times:

        max_after = maxima_times[maxima_times > t_min]
        if len(max_after) == 0:
            continue

        t_max = max_after[0]

        cycles_up.append((t_min, t_max))

    for t_max in maxima_times:

        min_after = minima_times[minima_times > t_max]
        if len(min_after) == 0:
            continue

        t_min = min_after[0]

        cycles_down.append((t_max, t_min))


    # ====== PHASE ASCENDANTE ======
    n_up = len(cycles_up)

    for i, (t_start, t_end) in enumerate(cycles_up):

        mask = (df_final[time_col] >= t_start) & (df_final[time_col] <= t_end)

        segment_time = df_final[time_col][mask].values - t_start
        segment_signal = df_final[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_up)

        ax_up.plot(segment_time, segment_signal, color=color)


    # ====== PHASE DESCENDANTE ======
    n_down = len(cycles_down)

    for i, (t_start, t_end) in enumerate(cycles_down):

        mask = (df_final[time_col] >= t_start) & (df_final[time_col] <= t_end)

        segment_time = df_final[time_col][mask].values - t_start
        segment_signal = df_final[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / n_down)

        ax_down.plot(segment_time, segment_signal, color=color)


    # ====== STYLE ======
    ax_up.set_xlabel("Temps depuis minima (s)")
    ax_up.set_ylabel(col)
    ax_up.set_title(f"Phases ascendantes : {col}")
    ax_up.grid(True)

    ax_down.set_xlabel("Temps depuis maxima (s)")
    ax_down.set_ylabel(col)
    ax_down.set_title(f"Phases descendantes : {col}")
    ax_down.grid(True)


    # ====== COLORBARS ======
    sm_up = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_up)))
    sm_up.set_array([])
    fig_up.colorbar(sm_up, ax=ax_up).set_label("Numéro cycle")

    sm_down = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_down)))
    sm_down.set_array([])
    fig_down.colorbar(sm_down, ax=ax_down).set_label("Numéro cycle")

    fig_up.tight_layout()
    fig_down.tight_layout()

    # ====== SAUVEGARDE ======
    fig_up.savefig(os.path.join(dossier_plots_sep, f"asc_{col}.png"), dpi=150)
    fig_down.savefig(os.path.join(dossier_plots_sep, f"desc_{col}.png"), dpi=150)

    plt.close(fig_up)
    plt.close(fig_down)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_plots_sep}")


✅ Graphiques sauvegardés dans : Cinematique_S04_S\superp_sep_S04


In [18]:
# ====== PARAMETRES ======
time_col = "Time"
cols = [c for c in velocities.columns if c != time_col]

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_plots_sep = os.path.join(dossier_root, "superv_sep_S04")
os.makedirs(dossier_plots_sep, exist_ok=True)

cmap = plt.cm.viridis

# ====== SUPERPOSITION ASCENDANT / DESCENDANT ======
for col in cols:

    if col not in minima_times_df.columns or col not in maxima_times_df.columns:
        continue

    minima_times = minima_times_df[col].dropna().values
    maxima_times = maxima_times_df[col].dropna().values

    if len(minima_times) < 1 or len(maxima_times) < 1:
        continue

    fig_up, ax_up = plt.subplots(figsize=(8,4))
    fig_down, ax_down = plt.subplots(figsize=(8,4))

    cycles_up = []
    cycles_down = []

    # ====== ASSOCIER MIN -> MAX ======
    for t_min in minima_times:

        max_after = maxima_times[maxima_times > t_min]
        if len(max_after) == 0:
            continue

        t_max = max_after[0]
        cycles_up.append((t_min, t_max))

    # ====== ASSOCIER MAX -> MIN ======
    for t_max in maxima_times:

        min_after = minima_times[minima_times > t_max]
        if len(min_after) == 0:
            continue

        t_min = min_after[0]
        cycles_down.append((t_max, t_min))


    # ====== PHASE ASCENDANTE ======
    n_up = len(cycles_up)

    for i, (t_start, t_end) in enumerate(cycles_up):

        mask = (velocities[time_col] >= t_start) & (velocities[time_col] <= t_end)

        segment_time = velocities[time_col][mask].values - t_start
        segment_signal = velocities[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / max(1, n_up))
        ax_up.plot(segment_time, segment_signal, color=color)


    # ====== PHASE DESCENDANTE ======
    n_down = len(cycles_down)

    for i, (t_start, t_end) in enumerate(cycles_down):

        mask = (velocities[time_col] >= t_start) & (velocities[time_col] <= t_end)

        segment_time = velocities[time_col][mask].values - t_start
        segment_signal = velocities[col][mask].values

        if len(segment_time) == 0:
            continue

        color = cmap(i / max(1, n_down))
        ax_down.plot(segment_time, segment_signal, color=color)


    # ====== STYLE ======
    ax_up.set_xlabel("Temps depuis minima (s)")
    ax_up.set_ylabel(f"Vitesse {col}")
    ax_up.set_title(f"Phases ascendantes : {col}")
    ax_up.grid(True)

    ax_down.set_xlabel("Temps depuis maxima (s)")
    ax_down.set_ylabel(f"Vitesse {col}")
    ax_down.set_title(f"Phases descendantes : {col}")
    ax_down.grid(True)


    # ====== COLORBARS ======
    sm_up = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_up)))
    sm_up.set_array([])
    fig_up.colorbar(sm_up, ax=ax_up).set_label("Numéro cycle")

    sm_down = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_down)))
    sm_down.set_array([])
    fig_down.colorbar(sm_down, ax=ax_down).set_label("Numéro cycle")

    fig_up.tight_layout()
    fig_down.tight_layout()


    # ====== SAUVEGARDE ======
    fig_up.savefig(os.path.join(dossier_plots_sep, f"asc_v_{col}.png"), dpi=150)
    fig_down.savefig(os.path.join(dossier_plots_sep, f"desc_v_{col}.png"), dpi=150)

    plt.close(fig_up)
    plt.close(fig_down)


print(f"\n✅ Graphiques de vitesses sauvegardés dans : {dossier_plots_sep}")


✅ Graphiques de vitesses sauvegardés dans : Cinematique_S04_S\superv_sep_S04


In [19]:
# ===== TEMPS A OBSERVER ========
t_min = 7
t_max = 37

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_save = os.path.join(dossier_root, "angle_manipu_S04")
os.makedirs(dossier_save, exist_ok=True)

time_col = "Time"

# ====== CONDITIONS DISPONIBLES ======
bases = sorted(set([c.split("_center")[0] for c in df_angle_base.columns if "_center_X" in c]))

# ====== BOUCLE SUR CHAQUE FICHIER ======
for base in bases:

    center_x = df_angle_base[f"{base}_center_X"].values
    center_y = df_angle_base[f"{base}_center_Y"].values

    top_x = df_angle_base[f"{base}_top_X"].values
    top_Z = df_angle_base[f"{base}_top_Z"].values

    # ===== vecteur centre → haut =====
    vx = top_x - center_x
    vy = top_z - center_y

    # ===== angle (positif vers gauche) =====
    angle = -np.degrees(np.arctan2(vx, vy))

    angle_df = pd.DataFrame({
        "Time": df_angle_base["Time"],
        "Angle": angle
    })

    # ===== SELECTION INTERVALLE TEMPS =====
    mask = (angle_df["Time"] >= t_min) & (angle_df["Time"] <= t_max)

    t = angle_df["Time"][mask].values
    a = angle_df["Angle"][mask].values

    if len(t) == 0:
        continue

    # ===== FIGURE =====
    fig, ax = plt.subplots(figsize=(8,4))

    ax.plot(t, a)

    # ===== STYLE =====
    ax.set_xlabel("Temps (s)")
    ax.set_ylabel("Angle du rectangle (°)")
    ax.set_title(f"Angle rectangle : {base}")
    ax.grid(True)

    fig.tight_layout()

    # ===== SAUVEGARDE =====
    save_path = os.path.join(dossier_save, f"angle_time_{base}.png")
    fig.savefig(save_path, dpi=150)

    plt.close(fig)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_save}")


✅ Graphiques sauvegardés dans : Cinematique_S04_S\angle_manipu_S04


In [20]:
# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_save = os.path.join(dossier_root, "supera_manipu_S04")
os.makedirs(dossier_save, exist_ok=True)

time_col = "Time"
cmap = plt.cm.viridis

# ====== CONDITIONS DISPONIBLES ======
bases = sorted(set([c.split("_center")[0] for c in df_angle_base.columns if "_center_X" in c]))

# ====== BOUCLE SUR CHAQUE FICHIER ======
for base in bases:

    if base+"_X" not in minima_times_df.columns:
        continue

    center_x = df_angle_base[f"{base}_center_X"].values
    center_y = df_angle_base[f"{base}_center_Y"].values

    top_x = df_angle_base[f"{base}_top_X"].values
    top_z = df_angle_base[f"{base}_top_Z"].values

    # ===== vecteur centre → haut =====
    vx = top_x - center_x
    vy = top_z - center_y

    # ===== angle (positif vers gauche) =====
    angle = -np.degrees(np.arctan2(vx, vy))

    angle_df = pd.DataFrame({
        "Time": df_angle_base["Time"],
        "Angle": angle
    })

    minima_times = minima_times_df[base+"_X"].dropna().values

    if len(minima_times) < 2:
        continue

    # ===== FIGURE =====
    fig, ax = plt.subplots(figsize=(8,4))

    n_cycles = len(minima_times) - 1

    for i in range(n_cycles):

        t_start = minima_times[i]
        t_end = minima_times[i+1]

        mask = (angle_df["Time"] >= t_start) & (angle_df["Time"] <= t_end)

        t = angle_df["Time"][mask].values
        a = angle_df["Angle"][mask].values

        if len(t) == 0:
            continue

        t_norm = (t - t_start) / (t_end - t_start)

        color = cmap(i / max(1,n_cycles))

        ax.plot(t_norm, a, color=color)

    # ===== STYLE =====
    ax.set_xlabel("Progression du cycle")
    ax.set_ylabel("Angle du rectangle (°)")
    ax.set_title(f"Superposition cycles angle : {base}")
    ax.grid(True)

    # ===== COLORBAR =====
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=1, vmax=max(1,n_cycles)))
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("Numéro cycle")

    fig.tight_layout()

    # ===== SAUVEGARDE =====
    save_path = os.path.join(dossier_save, f"angle_cycles_{base}.png")
    fig.savefig(save_path, dpi=150)

    plt.close(fig)

print(f"\n✅ Graphiques sauvegardés dans : {dossier_save}")


✅ Graphiques sauvegardés dans : Cinematique_S04_S\supera_manipu_S04


In [21]:
# ========================
# PARAMÈTRES
# ========================
npts_avg = 5

# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_angles = os.path.join(dossier_root, "angles_signes_s04")
os.makedirs(dossier_angles, exist_ok=True)

# ========================
# FONCTIONS
# ========================
def get_xz_bases(df):
    return sorted(
        {c[:-2] for c in df.columns if c.endswith("_X")}
        & {c[:-2] for c in df.columns if c.endswith("_Z")}
    )

def extract_cycles(mins, maxs):
    cycles = []
    for i in range(len(mins)-1):
        t_min1 = mins[i]
        t_min2 = mins[i+1]

        max_between = maxs[(maxs > t_min1) & (maxs < t_min2)]
        if len(max_between) == 0:
            continue

        t_max = max_between[0]
        cycles.append((t_min1, t_max, t_min2))
    return cycles

def angle_vs_vertical_signed(x0, y0, x1, y1):
    dx = -(x1 - x0)
    dy = abs(y1 - y0)
    return float(np.degrees(np.arctan2(dx, dy)))

def max_distance_to_line(x, y, x0, y0, x1, y1):
    vx, vy = x1 - x0, y1 - y0
    norm_v = np.hypot(vx, vy)
    if norm_v < 1e-12:
        return 0.0
    dist = np.abs((vy*(x - x0) - vx*(y - y0)) / norm_v)
    return float(np.max(dist))

# ========================
# DATAFRAME CENTRAL
# ========================
rows = []

xz_bases = get_xz_bases(df_final)

for base in xz_bases:

    colX = f"{base}_X"
    colZ = f"{base}_Z"

    if colZ not in minima_times_df.columns or colZ not in maxima_times_df.columns:
        continue

    mins = minima_times_df[colZ].dropna().values
    maxs = maxima_times_df[colZ].dropna().values

    cycles = extract_cycles(mins, maxs)

    for i, (t_min1, t_max, t_min2) in enumerate(cycles):

        # ========= ASC =========
        mask = (df_final["Time"] >= t_min1) & (df_final["Time"] <= t_max)
        x = df_final[colX][mask].values
        z = df_final[colZ][mask].values

        if len(x) >= 2*npts_avg+1:
            x0, z0 = np.mean(x[:npts_avg]), np.mean(z[:npts_avg])
            x1, z1 = np.mean(x[-npts_avg:]), np.mean(z[-npts_avg:])

            rows.append({
                "base": base,
                "cycle": i,
                "phase": "ASC",
                "t_start": t_min1,
                "t_end": t_max,
                "angle": angle_vs_vertical_signed(x0,z0,x1,z1),
                "dist": max_distance_to_line(x,z,x0,z0,x1,z1)
            })

        # ========= DESC =========
        mask = (df_final["Time"] >= t_max) & (df_final["Time"] <= t_min2)
        x = df_final[colX][mask].values
        z = df_final[colZ][mask].values

        if len(x) >= 2*npts_avg+1:
            x0, z0 = np.mean(x[:npts_avg]), np.mean(z[:npts_avg])
            x1, z1 = np.mean(x[-npts_avg:]), np.mean(z[-npts_avg:])

            rows.append({
                "base": base,
                "cycle": i,
                "phase": "DESC",
                "t_start": t_max,
                "t_end": t_min2,
                "angle": angle_vs_vertical_signed(x0,z0,x1,z1),
                "dist": max_distance_to_line(x,z,x0,z0,x1,z1)
            })

# Création DF
df_cycles = pd.DataFrame(rows)

print("✅ DataFrame central créé ")

print(f"✅ Paires XZ : {len(xz_bases)} trouvées.")

✅ DataFrame central créé 
✅ Paires XZ : 12 trouvées.


In [22]:
for base in df_cycles["base"].unique():

    df_b = df_cycles[df_cycles["base"] == base]

    for phase, color in [("ASC","blue"), ("DESC","orange")]:

        df_p = df_b[df_b["phase"] == phase]

        if len(df_p) == 0:
            continue

        # ANGLE
        plt.figure()
        plt.plot(df_p["cycle"], df_p["angle"], marker='o', color=color)
        plt.title(f"Angle {phase} – {base}")
        plt.grid()
        plt.savefig(os.path.join(dossier_angles, f"angle_{phase}_{base}.png"))
        plt.close()

        # DISTANCE
        plt.figure()
        plt.plot(df_p["cycle"], df_p["dist"], marker='o', color=color)
        plt.title(f"Dist {phase} – {base}")
        plt.grid()
        plt.savefig(os.path.join(dossier_angles, f"dist_{phase}_{base}.png"))
        plt.close()

    print(f"✅ Plots angles et dist enregistrés pour {base}")

✅ Plots angles et dist enregistrés pour S04_S1_13
✅ Plots angles et dist enregistrés pour S04_S1_14
✅ Plots angles et dist enregistrés pour S04_S1_15
✅ Plots angles et dist enregistrés pour S04_S2_16
✅ Plots angles et dist enregistrés pour S04_S2_17
✅ Plots angles et dist enregistrés pour S04_S2_18
✅ Plots angles et dist enregistrés pour S04_S3_19
✅ Plots angles et dist enregistrés pour S04_S3_20
✅ Plots angles et dist enregistrés pour S04_S3_21
✅ Plots angles et dist enregistrés pour S04_S4_22
✅ Plots angles et dist enregistrés pour S04_S4_23
✅ Plots angles et dist enregistrés pour S04_S4_24


In [23]:
# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_phase = os.path.join(dossier_root, "diagramme_phase_s04")
os.makedirs(dossier_phase, exist_ok=True)

for base in df_cycles["base"].unique():

    colX = f"{base}_X"
    colY = f"{base}_Y"

    df_b = df_cycles[df_cycles["base"] == base]

    X_full, Y_full = [], []

    for _, row in df_b.iterrows():
        mask = (df_final["Time"] >= row["t_start"]) & (df_final["Time"] <= row["t_end"])
        X_full.extend(df_final[colX][mask].values)
        Y_full.extend(df_final[colY][mask].values)

    plt.figure(figsize=(6,6))
    plt.plot(X_full, Y_full)
    plt.axis('equal')
    plt.grid()
    plt.title(f"Phase globale – {base}")
    plt.savefig(os.path.join(dossier_phase, f"{base}.png"))
    plt.close()
    
    print(f"✅ Diagrammes de phase complet enregistrés pour {base}")

✅ Diagrammes de phase complet enregistrés pour S04_S1_13
✅ Diagrammes de phase complet enregistrés pour S04_S1_14
✅ Diagrammes de phase complet enregistrés pour S04_S1_15
✅ Diagrammes de phase complet enregistrés pour S04_S2_16
✅ Diagrammes de phase complet enregistrés pour S04_S2_17
✅ Diagrammes de phase complet enregistrés pour S04_S2_18
✅ Diagrammes de phase complet enregistrés pour S04_S3_19
✅ Diagrammes de phase complet enregistrés pour S04_S3_20
✅ Diagrammes de phase complet enregistrés pour S04_S3_21
✅ Diagrammes de phase complet enregistrés pour S04_S4_22
✅ Diagrammes de phase complet enregistrés pour S04_S4_23
✅ Diagrammes de phase complet enregistrés pour S04_S4_24


In [24]:
# ====== DOSSIER DES PLOTS ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

dossier_phase_sep = os.path.join(dossier_root, "diagramme_phase_sep_s04")
os.makedirs(dossier_phase_sep, exist_ok=True)

for base in df_cycles["base"].unique():

    colX = f"{base}_X"
    colZ = f"{base}_Z"

    df_b = df_cycles[df_cycles["base"] == base]

    for phase, color in [("ASC","green"), ("DESC","red")]:

        df_p = df_b[df_b["phase"] == phase]

        X, Z = [], []

        for _, row in df_p.iterrows():
            mask = (df_final["Time"] >= row["t_start"]) & (df_final["Time"] <= row["t_end"])
            X.extend(df_final[colX][mask].values)
            Z.extend(df_final[colY][mask].values)

        if len(X) == 0:
            continue

        plt.figure(figsize=(6,6))
        plt.plot(X, Z, color=color)
        plt.axis('equal')
        plt.grid()
        plt.title(f"{phase} – {base}")
        plt.savefig(os.path.join(dossier_phase_sep, f"{phase}_{base}.png"))
        plt.close()

    print(f"✅ Diagrammes ASC et DESC pour {base}")

✅ Diagrammes ASC et DESC pour S04_S1_13
✅ Diagrammes ASC et DESC pour S04_S1_14
✅ Diagrammes ASC et DESC pour S04_S1_15
✅ Diagrammes ASC et DESC pour S04_S2_16
✅ Diagrammes ASC et DESC pour S04_S2_17
✅ Diagrammes ASC et DESC pour S04_S2_18
✅ Diagrammes ASC et DESC pour S04_S3_19
✅ Diagrammes ASC et DESC pour S04_S3_20
✅ Diagrammes ASC et DESC pour S04_S3_21
✅ Diagrammes ASC et DESC pour S04_S4_22
✅ Diagrammes ASC et DESC pour S04_S4_23
✅ Diagrammes ASC et DESC pour S04_S4_24


In [25]:
# ====== DOSSIER RACINE ======
dossier_root = "Cinematique_S04_S"
os.makedirs(dossier_root, exist_ok=True)

# ====== DOSSIER DES PLOTS SHIFT + MOYENNE ======
dossier_phase_shift = os.path.join(dossier_root, "diagramme_phase_shift_s04")
os.makedirs(dossier_phase_shift, exist_ok=True)

# ====== LISTE DES BASES XY ======
xy_bases = sorted(
    {c[:-2] for c in df_final.columns if c.endswith("_X")}
    & {c[:-2] for c in df_final.columns if c.endswith("_Z")}
)
print(f"✅ Paires XY détectées : {len(xy_bases)}")

# ====== BOUCLE SUR LES BASES ======
for base in xy_bases:

    colX = f"{base}_X"
    colZ = f"{base}_Z"

    if colZ not in minima_times_df.columns or colZ not in maxima_times_df.columns:
        print(f"⚠️ Pas d’extréma pour {base}, skip.")
        continue

    mins = minima_times_df[colY].dropna().values
    maxs = maxima_times_df[colY].dropna().values

    # ====== LISTES DE STOCKAGE DES CYCLES ======
    X_ASC_cycles = []
    Z_ASC_cycles = []
    X_DESC_cycles = []
    Z_DESC_cycles = []

    # ====== EXTRACTION DES CYCLES ======
    for i in range(len(mins)-1):

        t_min1 = mins[i]
        t_min2 = mins[i+1]

        max_between = maxs[(maxs > t_min1) & (maxs < t_min2)]
        if len(max_between) == 0:
            continue

        t_max = max_between[0]

        # --- Phase ASC ---
        mask_asc = (df_final["Time"] >= t_min1) & (df_final["Time"] <= t_max)
        Xa = df_final[colX][mask_asc].values
        Za = df_final[colY][mask_asc].values

        # Shift min à 0
        Xa_shift = Xa - Xa[0]
        Za_shift = Za - Za[0]

        X_ASC_cycles.append(Xa_shift)
        Z_ASC_cycles.append(Za_shift)

        # --- Phase DESC ---
        mask_desc = (df_final["Time"] >= t_max) & (df_final["Time"] <= t_min2)
        Xd = df_final[colX][mask_desc].values
        Zd = df_final[colZ][mask_desc].values

        Xd_shift = Xd - Xd[0]
        Zd_shift = Zd - Zd[0]

        X_DESC_cycles.append(Xd_shift)
        Z_DESC_cycles.append(Zd_shift)

    # ====== FONCTION POUR MOYENNE INTERPOLÉE ======
    def compute_mean_curve(list_X, list_Y, N=200):
        if len(list_X) == 0:
            return None, None
        t_common = np.linspace(0, 1, N)
        X_interp = []
        Y_interp = []
        for Xc, Yc in zip(list_X, list_Y):
            t_orig = np.linspace(0, 1, len(Xc))
            X_interp.append(np.interp(t_common, t_orig, Xc))
            Y_interp.append(np.interp(t_common, t_orig, Yc))
        X_mean = np.mean(np.array(X_interp), axis=0)
        Y_mean = np.mean(np.array(Y_interp), axis=0)
        return X_mean, Y_mean

    X_ASC_mean, Z_ASC_mean = compute_mean_curve(X_ASC_cycles, Z_ASC_cycles)
    X_DESC_mean, Z_DESC_mean = compute_mean_curve(X_DESC_cycles, Z_DESC_cycles)

    # ====== PLOT PHASE ASC ======
    if len(X_ASC_cycles) > 0:
        plt.figure(figsize=(6,6))
        # Cycles transparents
        for Xc, Zc in zip(X_ASC_cycles, Z_ASC_cycles):
            plt.plot(Xc, Zc, color='green', alpha=0.25)
        # Moyenne
        if X_ASC_mean is not None:
            plt.plot(X_ASC_mean, Z_ASC_mean, color='black', linewidth=2)
        plt.xlabel("X shifté")
        plt.ylabel("Z shifté (min=0)")
        plt.title(f"Phase ASC – Moyenne + cycles – {base}")
        plt.grid(True)
        plt.axis("equal")
        plt.tight_layout()
        plt.savefig(os.path.join(dossier_phase_shift, f"phase_ASC_shift_{base}.png"), dpi=150)
        plt.close()

    # ====== PLOT PHASE DESC ======
    if len(X_DESC_cycles) > 0:
        plt.figure(figsize=(6,6))
        for Xc, Zc in zip(X_DESC_cycles, Z_DESC_cycles):
            plt.plot(Xc, Zc, color='red', alpha=0.25)
        if X_DESC_mean is not None:
            plt.plot(X_DESC_mean, Z_DESC_mean, color='black', linewidth=2)
        plt.xlabel("X shifté")
        plt.ylabel("Z shifté (max=0)")
        plt.title(f"Phase DESC – Moyenne + cycles – {base}")
        plt.grid(True)
        plt.axis("equal")
        plt.tight_layout()
        plt.savefig(os.path.join(dossier_phase_shift, f"phase_DESC_shift_{base}.png"), dpi=150)
        plt.close()

    print(f"✅ Diagrammes SHIFT + MOYENNE enregistrés pour {base}")

✅ Paires XY détectées : 12
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S1_13
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S1_14
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S1_15
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S2_16
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S2_17
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S2_18
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S3_19
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S3_20
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S3_21
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S4_22
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S4_23
✅ Diagrammes SHIFT + MOYENNE enregistrés pour S04_S4_24
